# Customer Churn Prediction

Portfolio analysis using the IBM Telco Customer Churn dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..')
DATA_PATH = PROJECT_ROOT / 'data' / 'Telco-Customer-Churn.csv'
IMAGE_DIR = PROJECT_ROOT / 'images'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
RANDOM_STATE = 42

sns.set_theme(style='whitegrid', font_scale=1.05)

## Data Cleaning

In [ ]:
# =========================
# Data Loading
# =========================

# Load dataset
df = pd.read_csv(DATA_PATH)
customer_ids = df['customerID'].copy()


# =========================
# Data Cleaning
# =========================

# Clean target variable
df['Churn'] = df['Churn'].astype(str).str.strip()

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle missing values
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Remove customer identifier
df = df.drop(columns=['customerID'])

df.head()

## EDA

In [ ]:
# =========================
# Exploratory Data Analysis
# =========================

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Churn Distribution
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='Churn', hue='Churn', palette=['#2E86AB', '#D1495B'], legend=False)
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Customers')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'churn_distribution.png', dpi=300)
plt.show()

# Contract Type vs Churn Rate
contract_churn = (
    df.assign(ChurnBinary=(df['Churn'] == 'Yes').astype(int))
    .groupby('Contract', as_index=False)['ChurnBinary']
    .mean()
    .sort_values('ChurnBinary', ascending=False)
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=contract_churn, x='Contract', y='ChurnBinary', hue='Contract', palette=['#D1495B', '#F29E4C', '#2E86AB'], legend=False)
ax.set_title('Contract Type vs Churn Rate')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'contract_churn_rate.png', dpi=300)
plt.show()

## Model Training

In [ ]:
# =========================
# Feature Engineering
# =========================

# Binary encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# One-hot encode categorical variables
categorical_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']


# =========================
# Train-Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)


# =========================
# Model Training
# =========================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=8,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)

## Evaluation

In [ ]:
# =========================
# Model Evaluation
# =========================

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
}

pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'model_metrics.csv', index=False)
print(pd.DataFrame([metrics]).round(4))
print(confusion_matrix(y_test, y_pred))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#2E86AB', linewidth=2, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], color='#9A9A9A', linestyle='--', linewidth=1)
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'roc_curve.png', dpi=300)
plt.show()

## Business Insights

In [ ]:
# =========================
# Feature Importance Analysis
# =========================

feature_importance = (
    pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(10)
)

plt.figure(figsize=(9, 6))
ax = sns.barplot(data=feature_importance, x='importance', y='feature', hue='feature', palette='viridis', legend=False)
ax.set_title('Top 10 Feature Importances')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'feature_importance.png', dpi=300)
plt.show()

# Customer Risk Ranking
risk_ranking = (
    pd.DataFrame({'customerID': customer_ids, 'churn_probability': model.predict_proba(X)[:, 1]})
    .sort_values('churn_probability', ascending=False)
    .head(100)
)
risk_ranking.to_csv(OUTPUT_DIR / 'high_risk_customers.csv', index=False)

# Business Insights
top_features = ', '.join(feature_importance['feature'].head(5))
insights = f"""
Model Summary
-------------
ROC-AUC: {metrics['roc_auc']:.4f}
F1 Score: {metrics['f1_score']:.4f}

Key Churn Drivers
-----------------
Top predictive features: {top_features}.

Customer Segments At Risk
-------------------------
Month-to-month customers, short-tenure customers, and high monthly charge customers show elevated risk.

Business Recommendations
------------------------
1. Prioritize outreach for customers with the highest churn probability.
2. Move month-to-month customers toward annual contracts.
3. Strengthen onboarding during the first 12 months.
4. Review pricing and value perception for high-charge customers.
5. Use targeted retention offers around renewal and billing milestones.
""".strip()

with open(OUTPUT_DIR / 'business_insights.txt', 'w', encoding='utf-8') as file:
    file.write(insights)

print(insights)